In [ ]:
# Imports and the diagnostic's own settings: the market and window the threshold was calibrated on (the primary market at the headline window), the histogram bins, the candidate thresholds and the sparsity targets (needs only data/, no pipeline output).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from regime_networks import config
from regime_networks.data import cleaned_prices_path
from regime_networks.returns import load_log_returns, rolling_correlations
from regime_networks.networks import mu_offdiag
from regime_networks.plotting import set_publication_style, style_axis, format_date_axis, annotate_events, NEUTRAL_COLOURS

CALIBRATION_INDEX = config.PRIMARY_INDEX
WINDOW_LENGTH, WINDOW_SHIFT = config.HEADLINE_WINDOW
CALIBRATION_BINS = 400
CALIBRATION_CANDIDATES = (0.0, 0.95, 191)
CALIBRATION_TARGET = 0.05
SPARSITY_TARGETS = [0.10, CALIBRATION_TARGET, 0.01]
set_publication_style()

In [ ]:
# Rolling correlation matrices of the calibration market, clipped to [-1, 1] as the network construction clips them, with each window's end date and mean off-diagonal correlation.
log_returns = load_log_returns(cleaned_prices_path(CALIBRATION_INDEX))
matrices, dates, stock_names = rolling_correlations(log_returns, WINDOW_LENGTH, WINDOW_SHIFT)
matrices = [np.clip(C, -1.0, 1.0) for C in matrices]
dates = pd.to_datetime(dates)
mean_correlation = pd.Series([mu_offdiag(C) for C in matrices], index=dates)
print(f"{CALIBRATION_INDEX} T={WINDOW_LENGTH}: {len(matrices)} windows, N={len(stock_names)} stocks")

In [ ]:
# Every off-diagonal correlation pooled over the windows of each sample (all windows, and the training windows only, ending by the train/test cutoff), counted in equal-width bins over [-1, 1].
bin_edges = np.linspace(-1.0, 1.0, CALIBRATION_BINS + 1)
left_edges = bin_edges[:-1]
is_training = dates <= pd.Timestamp(config.TRAIN_TEST_CUTOFF)
samples = {"full": matrices, "pre_cutoff": [C for C, keep in zip(matrices, is_training) if keep]}
counts = {sample: sum(np.histogram(C[np.triu_indices_from(C, k=1)], bins=bin_edges)[0] for C in sample_matrices) for sample, sample_matrices in samples.items()}

In [ ]:
# Fraction of pooled pairs at or above each candidate threshold (candidates lie on bin edges, so this matches the network rule C >= threshold exactly), and the smallest candidate that at most each sparsity target of pairs reach.
candidates = np.linspace(*CALIBRATION_CANDIDATES).round(10)
exceedance = {sample: np.array([n[(left_edges > t) | np.isclose(left_edges, t)].sum() / n.sum() for t in candidates]) for sample, n in counts.items()}
derived = pd.DataFrame([{"sample": sample, "n_windows": len(samples[sample]), "sparsity_target": target, "derived_threshold": candidates[np.flatnonzero(exceedance[sample] <= target)[0]],
                         "chosen_threshold": config.CORRELATION_THRESHOLD, "fraction_exceeding_chosen": exceedance[sample][np.isclose(candidates, config.CORRELATION_THRESHOLD)][0]}
                        for sample in samples for target in SPARSITY_TARGETS])
print(derived.to_string(index=False))

In [ ]:
# Mean off-diagonal correlation over time, with the named market events marked: the market-wide correlation level the threshold has to work across.
fig, ax = plt.subplots(figsize=(6.9, 3.0))
ax.plot(dates, mean_correlation, color=NEUTRAL_COLOURS["primary"], linewidth=1.1)
annotate_events(ax, config.CRISIS_EVENTS, dates[0], dates[-1])
style_axis(ax, grid=True)
format_date_axis(ax, interval_years=2)
ax.set_xlabel("Date")
ax.set_ylabel(r"Mean off-diagonal correlation ($\mu$)")
fig.tight_layout()
plt.show()

In [ ]:
# Mark the chosen threshold on an axis with a labelled dashed vertical line.
def mark_chosen(ax):
    ax.axvline(config.CORRELATION_THRESHOLD, color=NEUTRAL_COLOURS["reference"], linestyle="--", linewidth=1.3, zorder=3)
    ax.text(config.CORRELATION_THRESHOLD, ax.get_ylim()[1] * 0.97, f"threshold = {config.CORRELATION_THRESHOLD:.2f}", rotation=90, color=NEUTRAL_COLOURS["reference"], fontsize=9, va="top", ha="center")

In [ ]:
# Density of every pairwise correlation pooled over all windows, with the chosen threshold in the right tail.
fig, ax = plt.subplots(figsize=(3.35, 3.2))
ax.plot(left_edges + (bin_edges[1] - bin_edges[0]) / 2, counts["full"] / counts["full"].sum() / (bin_edges[1] - bin_edges[0]), color=NEUTRAL_COLOURS["primary"], linewidth=1.2)
mark_chosen(ax)
style_axis(ax, grid=True)
ax.set_xlabel("Pairwise off-diagonal correlation")
ax.set_ylabel("Density (pooled)")
fig.tight_layout()
plt.show()

In [ ]:
# Fraction of pooled pairs reaching each candidate threshold, for all windows and for the training windows only, with the sparsity target and the chosen threshold marked.
for sample, title in (("full", "All windows"), ("pre_cutoff", f"Training windows only (to {config.TRAIN_TEST_CUTOFF})")):
    fig, ax = plt.subplots(figsize=(3.35, 3.2))
    ax.plot(candidates, exceedance[sample], color=NEUTRAL_COLOURS["primary"], linewidth=1.2)
    ax.axhline(CALIBRATION_TARGET, color=NEUTRAL_COLOURS["reference"], linestyle=":", linewidth=1.3, zorder=2)
    mark_chosen(ax)
    ax.text(candidates[-1], CALIBRATION_TARGET, f"{CALIBRATION_TARGET:.0%} target ", color=NEUTRAL_COLOURS["reference"], fontsize=9, va="bottom", ha="right")
    style_axis(ax, grid=True)
    ax.set_title(title)
    ax.set_xlabel("Candidate threshold")
    ax.set_ylabel("Fraction exceeding threshold")
    fig.tight_layout()
    plt.show()